### Data Preprocessing

#### Import

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
reviews = pd.read_csv("rotten_tomatoes_movie_reviews.csv")
movies = pd.read_csv("rotten_tomatoes_movies.csv")

#### Parse

In [21]:
# We will start parsing by removing duplicate movies and reviews
movies = movies.drop_duplicates(subset='id')

# We turn Box Office sales from strings with letters (e.g 'M' for million) into dollar amounts
movies['boxOffice'] = movies['boxOffice'].str.replace('$', '').str.replace('M', 'e6').str.replace('K', 'e3')
movies['boxOffice'] = pd.to_numeric(movies['boxOffice'], errors='coerce')

# Now we convert release dates and review dates from text to datetime
movies['releaseDateTheaters'] = pd.to_datetime(movies['releaseDateTheaters'], errors='coerce')
movies['releaseDateStreaming'] = pd.to_datetime(movies['releaseDateStreaming'], errors='coerce')

# The genre can be a list of multiple genres, so we take the first one as the movie’s main genre
movies['primaryGenre'] = movies['genre'].str.split(',').str[0]

# Define the difference in scores
movies['scoreGap'] = movies['audienceScore'] - movies['tomatoMeter']

movies[['title', 'tomatoMeter', 'audienceScore', 'scoreGap', 'primaryGenre', 'boxOffice']].head()

,title,tomatoMeter,audienceScore,scoreGap,primaryGenre,boxOffice
0,Space Zombie Bingo!,NaN,50.0,NaN,Comedy,NaN
1,The Green Grass,NaN,NaN,NaN,Drama,NaN
2,"Love, Lies",NaN,43.0,NaN,Drama,NaN
3,Sore Losers,NaN,60.0,NaN,Action,NaN
4,Dinosaur Island,NaN,70.0,NaN,Fantasy,NaN


In [22]:
# Removing duplicates
reviews = reviews.drop_duplicates(subset='reviewId')
reviews['creationDate'] = pd.to_datetime(reviews['creationDate'], errors='coerce')

# In the raw data, the critics' original scores are in different formats such as ("3/4", "8/10", "B+")
# We can convert the fraction ones to a [0-1] scale so they can be compared (other formats become NaN)
parts = reviews['originalScore'].str.split('/', expand=True)
reviews['scoreNorm'] = pd.to_numeric(parts[0], errors='coerce') / pd.to_numeric(parts[1], errors='coerce')
reviews.loc[reviews['scoreNorm'] > 1, 'scoreNorm'] = np.nan

reviews['isFresh'] = reviews['reviewState'] == 'fresh'

reviews[['originalScore', 'scoreNorm', 'reviewState', 'isFresh']].head()

,originalScore,scoreNorm,reviewState,isFresh
0,3.5/4,0.875,fresh,True
1,1/5,0.200,rotten,False
2,NaN,NaN,fresh,True
3,2.5/5,0.500,rotten,False
4,NaN,NaN,fresh,True


#### Organize

In [23]:
# Count each movie's reviews and its percentage of Fresh reviews
review_counts = reviews.groupby('id').agg(numReviews=('reviewId', 'count'), freshRate=('isFresh', 'mean')).reset_index()
movies = movies.merge(review_counts, on='id', how='left')
display(movies.head())

# Keep movies with both an audience and Tomatometer (critic) score
scored = movies.dropna(subset=['tomatoMeter', 'audienceScore'])
scored[['tomatoMeter', 'audienceScore', 'scoreGap', 'runtimeMinutes', 'numReviews']].describe()

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix,primaryGenre,scoreGap,numReviews,freshRate
0,space-zombie-bingo,Space Zombie Bingo!,50.0,NaN,NaN,NaN,NaT,2018-08-25,75.0,"Comedy, Horror, Sci-fi",English,George Ormrod,"George Ormrod,John Sabotta",NaN,NaN,NaN,Comedy,NaN,NaN,NaN
1,the_green_grass,The Green Grass,NaN,NaN,NaN,NaN,NaT,2020-02-11,114.0,Drama,English,Tiffany Edwards,Tiffany Edwards,NaN,NaN,NaN,Drama,NaN,NaN,NaN
2,love_lies,"Love, Lies",43.0,NaN,NaN,NaN,NaT,NaT,120.0,Drama,Korean,"Park Heung-Sik,Heung-Sik Park","Ha Young-Joon,Jeon Yun-su,Song Hye-jin",NaN,NaN,NaN,Drama,NaN,2.0,1.0
3,the_sore_losers_1997,Sore Losers,60.0,NaN,NaN,NaN,NaT,2020-10-23,90.0,"Action, Mystery & thriller",English,John Michael McCarthy,John Michael McCarthy,NaN,NaN,NaN,Action,NaN,NaN,NaN
4,dinosaur_island_2002,Dinosaur Island,70.0,NaN,NaN,NaN,NaT,2017-03-27,80.0,"Fantasy, Adventure, Animation",English,Will Meugniot,John Loy,NaN,NaN,NaN,Fantasy,NaN,NaN,NaN


,tomatoMeter,audienceScore,scoreGap,runtimeMinutes,numReviews
count,30052.000000,30052.000000,30052.000000,29367.000000,29854.000000
mean,64.525090,62.310229,-2.214861,101.506453,44.397836
std,28.045784,20.862795,22.745236,21.090280,61.756210
min,0.000000,0.000000,-100.000000,5.000000,1.000000
25%,43.000000,47.000000,-17.000000,90.000000,9.000000
50%,71.000000,65.000000,-4.000000,98.000000,19.000000
75%,88.000000,79.000000,11.000000,110.000000,51.000000
max,100.000000,100.000000,100.000000,584.000000,597.000000


### Data Exploration